In [1]:
import sys
sys.path.append("..")

import time
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report
)
import xgboost as xgb

from app.ml.data_loading import load_all_cicids2017
from app.services.spring_client import register_model, record_experiment_result

In [2]:
path = "../datasets/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv"
df = pd.read_csv(path)

print("Shape (rreshta, kolona):", df.shape)
print("\nEmrat e kolonave (te para 20):")
print(df.columns.tolist()[:20])
print("\nTipet e te dhenave:")
print(df.dtypes.value_counts())
print("\nShembull rreshtash:")
df.head()

Shape (rreshta, kolona): (529918, 79)

Emrat e kolonave (te para 20):
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min']

Tipet e te dhenave:
int64      54
float64    24
object      1
Name: count, dtype: int64

Shembull rreshtash:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
print(df.columns.tolist())

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [4]:
path2 = "../datasets/TrafficLabelling/Monday-WorkingHours.pcap_ISCX.csv"
df2 = pd.read_csv(path2, encoding="latin1", low_memory=False)

print("Shape:", df2.shape)
print(df2.columns.tolist())

Shape: (529918, 85)
['Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance'

In [5]:
df2.columns = df2.columns.str.strip()

print("Kolona te duplikuara pas strip():")
print(df2.columns[df2.columns.duplicated()].tolist())

print("\nShape pas pastrimit:", df2.shape)
print("\nListe e plote e kolonave (te pastruara):")
print(df2.columns.tolist())

Kolona te duplikuara pas strip():
[]

Shape pas pastrimit: (529918, 85)

Liste e plote e kolonave (te pastruara):
['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length

In [6]:
identical = (df2['Fwd Header Length'] == df2['Fwd Header Length.1']).all()
print("A jane identike vlerat:", identical)

if not identical:
    diff_count = (df2['Fwd Header Length'] != df2['Fwd Header Length.1']).sum()
    print(f"Numri i rreshtave ku ndryshojne: {diff_count}")
    print(df2[['Fwd Header Length', 'Fwd Header Length.1']].head(10))

A jane identike vlerat: True


In [7]:
df2 = df2.drop(columns=['Fwd Header Length.1'])

print("Shape pas heqjes se duplikatit:", df2.shape)
print("\nA ka NaN/vlera mungese?")
print(df2.isnull().sum().sum(), "vlera NaN gjithsej")

print("\nA ka Infinity?")
numeric_cols = df2.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df2[numeric_cols]).sum().sum()
print(inf_count, "vlera Infinity gjithsej")

print("\nShperndarja e Label:")
print(df2['Label'].value_counts())

Shape pas heqjes se duplikatit: (529918, 84)

A ka NaN/vlera mungese?
64 vlera NaN gjithsej

A ka Infinity?
810 vlera Infinity gjithsej

Shperndarja e Label:
Label
BENIGN    529918
Name: count, dtype: int64


In [8]:
print("NaN sipas kolonave:")
print(df2.isnull().sum()[df2.isnull().sum() > 0])

print("\nInfinity sipas kolonave:")
inf_per_col = np.isinf(df2[numeric_cols]).sum()
print(inf_per_col[inf_per_col > 0])

NaN sipas kolonave:
Flow Bytes/s    64
dtype: int64

Infinity sipas kolonave:
Flow Bytes/s      373
Flow Packets/s    437
dtype: int64


In [9]:
before = len(df2)

df2 = df2.replace([np.inf, -np.inf], np.nan)
df2 = df2.dropna(subset=['Flow Bytes/s', 'Flow Packets/s'])

after = len(df2)

print(f"Rreshta para: {before}")
print(f"Rreshta pas: {after}")
print(f"U hoqen: {before - after} rreshta ({(before-after)/before*100:.3f}%)")

print("\nNaN mbetur gjithsej:", df2.isnull().sum().sum())
print("Infinity mbetur gjithsej:", np.isinf(df2.select_dtypes(include=[np.number])).sum().sum())

Rreshta para: 529918
Rreshta pas: 529481
U hoqen: 437 rreshta (0.082%)

NaN mbetur gjithsej: 0
Infinity mbetur gjithsej: 0


In [10]:
dataset_path = "../datasets/TrafficLabelling"
df_all = load_all_cicids2017(dataset_path)

print(f"\nGjithsej pas bashkimit: {len(df_all):,} rreshta, {df_all.shape[1]} kolona")
print("\nShperndarja e Label ne te gjithe dataset-in:")
print(df_all['Label'].value_counts())

  Monday-WorkingHours.pcap_ISCX.csv: 529,481 rreshta (pas pastrimit)
  Tuesday-WorkingHours.pcap_ISCX.csv: 445,645 rreshta (pas pastrimit)
  Wednesday-workingHours.pcap_ISCX.csv: 691,406 rreshta (pas pastrimit)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,231 rreshta (pas pastrimit)
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,395 rreshta (pas pastrimit)
  Friday-WorkingHours-Morning.pcap_ISCX.csv: 190,911 rreshta (pas pastrimit)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,096 rreshta (pas pastrimit)
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,711 rreshta (pas pastrimit)

Gjithsej pas bashkimit: 2,827,876 rreshta, 85 kolona

Shperndarja e Label ne te gjithe dataset-in:
Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator          

In [12]:
df_all.to_parquet("../datasets/cicids2017_cleaned.parquet")
print("U ruajt si parquet - here tjeter perdor pd.read_parquet(...) ne vend te load_all_cicids2017(...)")

U ruajt si parquet - here tjeter perdor pd.read_parquet(...) ne vend te load_all_cicids2017(...)


In [13]:
FEATURE_EXCLUDE = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP',
                    'Timestamp', 'Label', 'source_file']

X = df_all.drop(columns=FEATURE_EXCLUDE)
y = df_all['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nShperndarja Label ne train:")
print(y_train.value_counts())
print("\nShperndarja Label ne test:")
print(y_test.value_counts())

Train shape: (2262300, 78)
Test shape: (565576, 78)

Shperndarja Label ne train:
Label
BENIGN                        1817055
DoS Hulk                       184099
PortScan                       127043
DDoS                           102420
DoS GoldenEye                    8234
FTP-Patator                      6348
SSH-Patator                      4717
DoS slowloris                    4637
DoS Slowhttptest                 4399
Bot                              1565
Web Attack  Brute Force         1206
Web Attack  XSS                  522
Infiltration                       29
Web Attack  Sql Injection         17
Heartbleed                          9
Name: count, dtype: int64

Shperndarja Label ne test:
Label
BENIGN                        454265
DoS Hulk                       46025
PortScan                       31761
DDoS                           25605
DoS GoldenEye                   2059
FTP-Patator                     1587
SSH-Patator                     1180
DoS slowloris           

In [14]:
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

start = time.time()
rf_baseline.fit(X_train, y_train)
train_time = time.time() - start

start = time.time()
y_pred = rf_baseline.predict(X_test)
predict_time = time.time() - start

print(f"Koha e trajnimit: {train_time:.1f} sekonda")
print(f"Koha e parashikimit (gjithe test set): {predict_time:.2f} sekonda")
print(f"Koha mesatare per flow: {(predict_time/len(X_test))*1000:.4f} ms\n")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score (macro avg): {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-score (weighted avg): {f1_score(y_test, y_pred, average='weighted'):.4f}\n")

print("Raport i plote per cdo klase:")
print(classification_report(y_test, y_pred))

Koha e trajnimit: 374.8 sekonda
Koha e parashikimit (gjithe test set): 3.04 sekonda
Koha mesatare per flow: 0.0054 ms

Accuracy: 0.9986
F1-score (macro avg): 0.8724
F1-score (weighted avg): 0.9986

Raport i plote per cdo klase:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    454265
                       Bot       0.90      0.75      0.82       391
                      DDoS       1.00      1.00      1.00     25605
             DoS GoldenEye       1.00      0.99      1.00      2059
                  DoS Hulk       1.00      1.00      1.00     46025
          DoS Slowhttptest       0.99      0.99      0.99      1100
             DoS slowloris       1.00      1.00      1.00      1159
               FTP-Patator       1.00      1.00      1.00      1587
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.57      0.73         7
                  PortS

In [15]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)
model_path = models_dir / "rf_baseline_cicids2017_v1.joblib"
joblib.dump(rf_baseline, model_path)

print(f"Modeli u ruajt: {model_path}")
print(f"Madhesia: {model_path.stat().st_size / (1024*1024):.1f} MB")

Modeli u ruajt: ..\models\rf_baseline_cicids2017_v1.joblib
Madhesia: 148.4 MB


In [16]:
model_response = register_model(
    algorithm="RandomForest",
    name="rf-baseline-cicids2017-v1",
    trained_on_dataset="CICIDS2017",
    artifact_path="models/rf_baseline_cicids2017_v1.joblib",
    trained_at_iso=datetime.now(timezone.utc).isoformat(),
    hyperparameters='{"n_estimators": 100, "random_state": 42, "class_weight": null}',
    feature_set="all_78_features_excluding_identifiers"
)

print("Modeli u regjistrua:")
print(model_response)

model_id = model_response["id"]

HTTPError: 500 Server Error:  for url: http://localhost:8080/api/models

In [17]:
model_response = register_model(
    algorithm="RandomForest",
    name="rf-baseline-cicids2017-v2",
    trained_on_dataset="CICIDS2017",
    artifact_path="models/rf_baseline_cicids2017_v1.joblib",
    trained_at_iso=datetime.now(timezone.utc).isoformat(),
    hyperparameters='{"n_estimators": 100, "random_state": 42, "class_weight": null}',
    feature_set="all_78_features_excluding_identifiers"
)

print("Modeli u regjistrua:")
print(model_response)

model_id = model_response["id"]

Modeli u regjistrua:
{'id': 'f3a8cdd7-87cd-468a-861e-e301d41c356d', 'algorithm': 'RandomForest', 'name': 'rf-baseline-cicids2017-v2', 'trainedOnDataset': 'CICIDS2017', 'artifactPath': 'models/rf_baseline_cicids2017_v1.joblib', 'hyperparameters': '{"n_estimators": 100, "random_state": 42, "class_weight": null}', 'featureSet': 'all_78_features_excluding_identifiers', 'active': False, 'trainedAt': '2026-08-12T21:59:29.698584Z', 'createdAt': '2026-08-12T21:59:29.723324900Z'}


In [18]:
precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)

print(f"Precision (macro): {precision_macro:.4f}")
print(f"Recall (macro): {recall_macro:.4f}")

Precision (macro): 0.9381
Recall (macro): 0.8466


In [19]:
experiment_response = record_experiment_result(
    ml_model_id=model_id,
    tested_on_dataset="CICIDS2017",
    accuracy=0.9986,
    precision=precision_macro,
    recall=recall_macro,
    f1=0.8724,
    avg_latency_ms=0.0054,
    sample_size=len(X_test),
    feature_set_used="all_78_features_excluding_identifiers",
    notes="Baseline Random Forest v2, pa SMOTE, class_weight=None, 100 estimators"
)

print("Eksperimenti u regjistrua:")
print(experiment_response)


Eksperimenti u regjistrua:
{'id': '7ff62286-75bd-4449-99e4-3a147db92be6', 'mlModelId': 'f3a8cdd7-87cd-468a-861e-e301d41c356d', 'mlModelName': 'rf-baseline-cicids2017-v2', 'testedOnDataset': 'CICIDS2017', 'featureSetUsed': 'all_78_features_excluding_identifiers', 'accuracy': 0.9986, 'precisionScore': 0.9381494099629583, 'recall': 0.8466290313535187, 'f1Score': 0.8724, 'avgLatencyMs': 0.0054, 'sampleSize': 565576, 'notes': 'Baseline Random Forest v2, pa SMOTE, class_weight=None, 100 estimators', 'ranAt': '2026-08-12T22:00:10.280625100Z'}


In [20]:
!pip install dill

   ---------------------------------------- 0.0/120.0 kB ? eta -:--:--
   --- ------------------------------------ 10.2/120.0 kB ? eta -:--:--
   ------------- ------------------------- 41.0/120.0 kB 487.6 kB/s eta 0:00:01
   ---------------------------------------- 120.0/120.0 kB 1.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import dill

dill.dump_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u ruajt me sukses!")

Sesioni u ruajt me sukses!


In [2]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye - te gjitha variablat jane gati!")

Sesioni u rikthye - te gjitha variablat jane gati!


In [3]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Klasat (indeksi -> emri):")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i}: {label}")

Klasat (indeksi -> emri):
  0: BENIGN
  1: Bot
  2: DDoS
  3: DoS GoldenEye
  4: DoS Hulk
  5: DoS Slowhttptest
  6: DoS slowloris
  7: FTP-Patator
  8: Heartbleed
  9: Infiltration
  10: PortScan
  11: SSH-Patator
  12: Web Attack  Brute Force
  13: Web Attack  Sql Injection
  14: Web Attack  XSS


In [4]:
xgb_baseline = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    eval_metric='mlogloss'
)

start = time.time()
xgb_baseline.fit(X_train, y_train_encoded)
xgb_train_time = time.time() - start

start = time.time()
y_pred_xgb_encoded = xgb_baseline.predict(X_test)
xgb_predict_time = time.time() - start

xgb_accuracy = accuracy_score(y_test_encoded, y_pred_xgb_encoded)
xgb_f1_macro = f1_score(y_test_encoded, y_pred_xgb_encoded, average='macro')
xgb_precision_macro = precision_score(y_test_encoded, y_pred_xgb_encoded, average='macro', zero_division=0)
xgb_recall_macro = recall_score(y_test_encoded, y_pred_xgb_encoded, average='macro', zero_division=0)
xgb_latency_ms = (xgb_predict_time / len(X_test)) * 1000

print(f"Koha e trajnimit: {xgb_train_time:.1f} sekonda")
print(f"Koha e parashikimit: {xgb_predict_time:.2f} sekonda")
print(f"Koha mesatare per flow: {xgb_latency_ms:.4f} ms\n")

print(f"Accuracy: {xgb_accuracy:.4f}")
print(f"F1-score (macro avg): {xgb_f1_macro:.4f}")
print(f"Precision (macro): {xgb_precision_macro:.4f}")
print(f"Recall (macro): {xgb_recall_macro:.4f}\n")

print("Raport i plote per cdo klase:")
print(classification_report(y_test_encoded, y_pred_xgb_encoded, target_names=label_encoder.classes_))

Koha e trajnimit: 352.4 sekonda
Koha e parashikimit: 3.16 sekonda
Koha mesatare per flow: 0.0056 ms

Accuracy: 0.9990
F1-score (macro avg): 0.8811
Precision (macro): 0.9136
Recall (macro): 0.8629

Raport i plote per cdo klase:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    454265
                       Bot       0.91      0.79      0.84       391
                      DDoS       1.00      1.00      1.00     25605
             DoS GoldenEye       1.00      1.00      1.00      2059
                  DoS Hulk       1.00      1.00      1.00     46025
          DoS Slowhttptest       0.99      0.99      0.99      1100
             DoS slowloris       1.00      1.00      1.00      1159
               FTP-Patator       1.00      1.00      1.00      1587
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.57      0.73         7
                  PortSc

In [5]:
xgb_path = models_dir / "xgb_baseline_cicids2017_v1.joblib"
joblib.dump(xgb_baseline, xgb_path)
joblib.dump(label_encoder, models_dir / "label_encoder_cicids2017.joblib")

print(f"XGBoost u ruajt: {xgb_path}")
print(f"Label encoder u ruajt.")

XGBoost u ruajt: ..\models\xgb_baseline_cicids2017_v1.joblib
Label encoder u ruajt.


In [6]:
xgb_model_response = register_model(
    algorithm="XGBoost",
    name="xgb-baseline-cicids2017-v1",
    trained_on_dataset="CICIDS2017",
    artifact_path="models/xgb_baseline_cicids2017_v1.joblib",
    trained_at_iso=datetime.now(timezone.utc).isoformat(),
    hyperparameters='{"n_estimators": 100, "random_state": 42, "tree_method": "hist"}',
    feature_set="all_78_features_excluding_identifiers"
)
xgb_model_id = xgb_model_response["id"]
print("XGBoost model ID:", xgb_model_id)

xgb_experiment_response = record_experiment_result(
    ml_model_id=xgb_model_id,
    tested_on_dataset="CICIDS2017",
    accuracy=xgb_accuracy,
    precision=xgb_precision_macro,
    recall=xgb_recall_macro,
    f1=xgb_f1_macro,
    avg_latency_ms=xgb_latency_ms,
    sample_size=len(X_test),
    feature_set_used="all_78_features_excluding_identifiers",
    notes="Baseline XGBoost, pa SMOTE, 100 estimators, tree_method=hist"
)
print("XGBoost experiment registered:", xgb_experiment_response["id"])

XGBoost model ID: 9ff23608-75dc-4816-a651-c2f1e28d5a3c
XGBoost experiment registered: 91e2cb6c-79ed-4662-be60-ae311f2ccfa1


In [7]:
dill.dump_session('../notebooks/session_checkpoint.pkl')
print("Checkpoint u perditesua me XGBoost te perfshire!")

Checkpoint u perditesua me XGBoost te perfshire!


In [8]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

print("Shperndarja PARA balancimit:")
print(Counter(y_train))

# Strategjia: redukto BENIGN ne 200,000, ngrit klasat e rralla ne minimum 2,000
undersample_strategy = {'BENIGN': 200000}
oversample_strategy = {cls: 2000 for cls, count in Counter(y_train).items()
                        if count < 2000 and cls != 'BENIGN'}

print("\nKlasat qe do te SMOTE-ohen (nen 2000 shembuj):")
print(oversample_strategy)

pipeline = ImbPipeline([
    ('under', RandomUnderSampler(sampling_strategy=undersample_strategy, random_state=42)),
    ('over', SMOTE(sampling_strategy=oversample_strategy, random_state=42, k_neighbors=5))
])

X_train_balanced, y_train_balanced = pipeline.fit_resample(X_train, y_train)

print("\nShperndarja PAS balancimit:")
print(Counter(y_train_balanced))
print(f"\nShape i ri: {X_train_balanced.shape}")

Shperndarja PARA balancimit:
Counter({'BENIGN': 1817055, 'DoS Hulk': 184099, 'PortScan': 127043, 'DDoS': 102420, 'DoS GoldenEye': 8234, 'FTP-Patator': 6348, 'SSH-Patator': 4717, 'DoS slowloris': 4637, 'DoS Slowhttptest': 4399, 'Bot': 1565, 'Web Attack \x96 Brute Force': 1206, 'Web Attack \x96 XSS': 522, 'Infiltration': 29, 'Web Attack \x96 Sql Injection': 17, 'Heartbleed': 9})

Klasat qe do te SMOTE-ohen (nen 2000 shembuj):
{'Web Attack \x96 Brute Force': 2000, 'Web Attack \x96 XSS': 2000, 'Bot': 2000, 'Web Attack \x96 Sql Injection': 2000, 'Infiltration': 2000, 'Heartbleed': 2000}


C:\IDS-Service\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\IDS-Service\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\orgit\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\orgit\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\orgit\AppData\Local\Programs\Python\Python311\Lib\sub


Shperndarja PAS balancimit:
Counter({'BENIGN': 200000, 'DoS Hulk': 184099, 'PortScan': 127043, 'DDoS': 102420, 'DoS GoldenEye': 8234, 'FTP-Patator': 6348, 'SSH-Patator': 4717, 'DoS slowloris': 4637, 'DoS Slowhttptest': 4399, 'Bot': 2000, 'Heartbleed': 2000, 'Infiltration': 2000, 'Web Attack \x96 Brute Force': 2000, 'Web Attack \x96 Sql Injection': 2000, 'Web Attack \x96 XSS': 2000})

Shape i ri: (653897, 78)


In [9]:
rf_smote = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

start = time.time()
rf_smote.fit(X_train_balanced, y_train_balanced)
rf_smote_train_time = time.time() - start

start = time.time()
y_pred_rf_smote = rf_smote.predict(X_test)
rf_smote_predict_time = time.time() - start

rf_smote_accuracy = accuracy_score(y_test, y_pred_rf_smote)
rf_smote_f1_macro = f1_score(y_test, y_pred_rf_smote, average='macro')
rf_smote_precision_macro = precision_score(y_test, y_pred_rf_smote, average='macro', zero_division=0)
rf_smote_recall_macro = recall_score(y_test, y_pred_rf_smote, average='macro', zero_division=0)
rf_smote_latency_ms = (rf_smote_predict_time / len(X_test)) * 1000

print(f"Koha e trajnimit: {rf_smote_train_time:.1f} sekonda")
print(f"Koha mesatare per flow: {rf_smote_latency_ms:.4f} ms\n")

print(f"Accuracy: {rf_smote_accuracy:.4f}")
print(f"F1-score (macro avg): {rf_smote_f1_macro:.4f}")
print(f"Precision (macro): {rf_smote_precision_macro:.4f}")
print(f"Recall (macro): {rf_smote_recall_macro:.4f}\n")

print("Raport i plote per cdo klase:")
print(classification_report(y_test, y_pred_rf_smote))

Koha e trajnimit: 39.1 sekonda
Koha mesatare per flow: 0.0051 ms

Accuracy: 0.9985
F1-score (macro avg): 0.8646
Precision (macro): 0.8534
Recall (macro): 0.8792

Raport i plote per cdo klase:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    454265
                       Bot       0.71      0.95      0.81       391
                      DDoS       1.00      1.00      1.00     25605
             DoS GoldenEye       0.99      0.99      0.99      2059
                  DoS Hulk       1.00      1.00      1.00     46025
          DoS Slowhttptest       0.99      0.99      0.99      1100
             DoS slowloris       0.99      1.00      1.00      1159
               FTP-Patator       1.00      1.00      1.00      1587
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       0.57      0.57      0.57         7
                  PortScan       0.99      1.00      1.00  

In [10]:
rf_smote_path = models_dir / "rf_smote_cicids2017_v1.joblib"
joblib.dump(rf_smote, rf_smote_path)

rf_smote_model_response = register_model(
    algorithm="RandomForest",
    name="rf-smote-cicids2017-v1",
    trained_on_dataset="CICIDS2017",
    artifact_path="models/rf_smote_cicids2017_v1.joblib",
    trained_at_iso=datetime.now(timezone.utc).isoformat(),
    hyperparameters='{"n_estimators": 100, "random_state": 42, "smote_strategy": "undersample_benign_200k_oversample_rare_2k"}',
    feature_set="all_78_features_excluding_identifiers"
)
rf_smote_model_id = rf_smote_model_response["id"]
print("RF+SMOTE model ID:", rf_smote_model_id)

rf_smote_experiment_response = record_experiment_result(
    ml_model_id=rf_smote_model_id,
    tested_on_dataset="CICIDS2017",
    accuracy=rf_smote_accuracy,
    precision=rf_smote_precision_macro,
    recall=rf_smote_recall_macro,
    f1=rf_smote_f1_macro,
    avg_latency_ms=rf_smote_latency_ms,
    sample_size=len(X_test),
    feature_set_used="all_78_features_excluding_identifiers",
    notes="Random Forest me SMOTE+undersampling (BENIGN->200k, rare classes->2k min)"
)
print("RF+SMOTE experiment registered:", rf_smote_experiment_response["id"])

RF+SMOTE model ID: a4e1224f-664c-4820-a54c-2b01332ad24b
RF+SMOTE experiment registered: 9e1fdeb2-59bd-4dd7-b399-7134fa39f14e


In [11]:
dill.dump_session('../notebooks/session_checkpoint.pkl')
print("Checkpoint u perditesua!")


Checkpoint u perditesua!


In [12]:
y_train_balanced_encoded = label_encoder.transform(y_train_balanced)

xgb_smote = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    eval_metric='mlogloss'
)

start = time.time()
xgb_smote.fit(X_train_balanced, y_train_balanced_encoded)
xgb_smote_train_time = time.time() - start

start = time.time()
y_pred_xgb_smote_encoded = xgb_smote.predict(X_test)
xgb_smote_predict_time = time.time() - start

xgb_smote_accuracy = accuracy_score(y_test_encoded, y_pred_xgb_smote_encoded)
xgb_smote_f1_macro = f1_score(y_test_encoded, y_pred_xgb_smote_encoded, average='macro')
xgb_smote_precision_macro = precision_score(y_test_encoded, y_pred_xgb_smote_encoded, average='macro', zero_division=0)
xgb_smote_recall_macro = recall_score(y_test_encoded, y_pred_xgb_smote_encoded, average='macro', zero_division=0)
xgb_smote_latency_ms = (xgb_smote_predict_time / len(X_test)) * 1000

print(f"Koha e trajnimit: {xgb_smote_train_time:.1f} sekonda")
print(f"Accuracy: {xgb_smote_accuracy:.4f}")
print(f"F1-score (macro avg): {xgb_smote_f1_macro:.4f}")
print(f"Precision (macro): {xgb_smote_precision_macro:.4f}")
print(f"Recall (macro): {xgb_smote_recall_macro:.4f}\n")

print(classification_report(y_test_encoded, y_pred_xgb_smote_encoded, target_names=label_encoder.classes_))

Koha e trajnimit: 62.3 sekonda
Accuracy: 0.9988
F1-score (macro avg): 0.8785
Precision (macro): 0.8671
Recall (macro): 0.8978

                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    454265
                       Bot       0.70      0.99      0.82       391
                      DDoS       1.00      1.00      1.00     25605
             DoS GoldenEye       1.00      1.00      1.00      2059
                  DoS Hulk       1.00      1.00      1.00     46025
          DoS Slowhttptest       0.99      0.99      0.99      1100
             DoS slowloris       0.99      1.00      1.00      1159
               FTP-Patator       1.00      1.00      1.00      1587
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       0.71      0.71      0.71         7
                  PortScan       0.99      1.00      1.00     31761
               SSH-Patator       1.00      1.00      1.0

In [13]:
xgb_smote_path = models_dir / "xgb_smote_cicids2017_v1.joblib"
joblib.dump(xgb_smote, xgb_smote_path)

xgb_smote_model_response = register_model(
    algorithm="XGBoost",
    name="xgb-smote-cicids2017-v1",
    trained_on_dataset="CICIDS2017",
    artifact_path="models/xgb_smote_cicids2017_v1.joblib",
    trained_at_iso=datetime.now(timezone.utc).isoformat(),
    hyperparameters='{"n_estimators": 100, "random_state": 42, "tree_method": "hist", "smote_strategy": "undersample_benign_200k_oversample_rare_2k"}',
    feature_set="all_78_features_excluding_identifiers"
)
xgb_smote_model_id = xgb_smote_model_response["id"]
print("XGBoost+SMOTE model ID:", xgb_smote_model_id)

xgb_smote_experiment_response = record_experiment_result(
    ml_model_id=xgb_smote_model_id,
    tested_on_dataset="CICIDS2017",
    accuracy=xgb_smote_accuracy,
    precision=xgb_smote_precision_macro,
    recall=xgb_smote_recall_macro,
    f1=xgb_smote_f1_macro,
    avg_latency_ms=xgb_smote_latency_ms,
    sample_size=len(X_test),
    feature_set_used="all_78_features_excluding_identifiers",
    notes="XGBoost me SMOTE+undersampling (BENIGN->200k, rare classes->2k min)"
)
print("XGBoost+SMOTE experiment registered:", xgb_smote_experiment_response["id"])

XGBoost+SMOTE model ID: d539a59a-4119-42c3-824e-bad9720cd35b
XGBoost+SMOTE experiment registered: 4eee2d7e-d8f3-48e3-a583-6a1e6987e667


In [14]:
dill.dump_session('../notebooks/session_checkpoint.pkl')
print("Checkpoint u perditesua me te 4 modelet!")

Checkpoint u perditesua me te 4 modelet!


In [ ]:
dill.dump_session('../notebooks/session_checkpoint.pkl')
print("Checkpoint u ruajt - gati per neser!")

In [1]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye - te gjitha variablat jane gati!")

Sesioni u rikthye - te gjitha variablat jane gati!


In [2]:
import shap

sample_size = 500
X_sample = X_test.sample(n=sample_size, random_state=42)

explainer = shap.TreeExplainer(xgb_smote)
shap_values = explainer.shap_values(X_sample)

print(f"SHAP values u llogariten per {sample_size} shembuj")
print(f"Shape i shap_values: {np.array(shap_values).shape}")

SHAP values u llogariten per 500 shembuj
Shape i shap_values: (500, 78, 15)


In [3]:
# Mesatarja e |SHAP values| neper te gjitha klasat, per te pare rendesine e pergjithshme te features
shap_values_array = np.array(shap_values)  # (500, 78, 15)
mean_abs_shap = np.abs(shap_values_array).mean(axis=(0, 2))  # mesatare mbi shembuj DHE klasa

feature_importance = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("Top 15 features me te rendesishme (sipas SHAP):")
print(feature_importance.head(15))

Top 15 features me te rendesishme (sipas SHAP):
                    feature  mean_abs_shap
0          Destination Port       1.220542
67  Init_Win_bytes_backward       0.581247
66   Init_Win_bytes_forward       0.469391
69     min_seg_size_forward       0.416498
38            Bwd Packets/s       0.415002
20             Flow IAT Min       0.258188
25              Fwd IAT Min       0.220082
7     Fwd Packet Length Max       0.219322
17            Flow IAT Mean       0.162376
24              Fwd IAT Max       0.158007
47           PSH Flag Count       0.153434
22             Fwd IAT Mean       0.132235
2             Flow Duration       0.129481
35        Fwd Header Length       0.128707
68         act_data_pkt_fwd       0.122554


In [4]:
# Gjej nje shembull konkret qe eshte ATTACK ne test set (jo BENIGN)
attack_indices = y_test[y_test != 'BENIGN'].index
sample_idx = attack_indices[0]  # merr te parin per shembull

sample_row = X_test.loc[[sample_idx]]
true_label = y_test.loc[sample_idx]
predicted_label_idx = xgb_smote.predict(sample_row)[0]
predicted_label = label_encoder.inverse_transform([predicted_label_idx])[0]

print(f"Etiketa e vertete: {true_label}")
print(f"Parashikimi i modelit: {predicted_label}")

# SHAP per kete shembull specifik
single_shap = explainer.shap_values(sample_row)
single_shap_for_predicted_class = single_shap[0, :, predicted_label_idx]

top_features = pd.DataFrame({
    'feature': sample_row.columns,
    'value': sample_row.values[0],
    'shap_contribution': single_shap_for_predicted_class
}).sort_values('shap_contribution', key=abs, ascending=False)

print(f"\nTop 5 features qe shtyne drejt parashikimit '{predicted_label}':")
print(top_features.head(5))

Etiketa e vertete: DDoS
Parashikimi i modelit: DDoS

Top 5 features qe shtyne drejt parashikimit 'DDoS':
                        feature  value  shap_contribution
68             act_data_pkt_fwd    4.0           2.398114
5   Total Length of Fwd Packets   30.0           1.095304
66       Init_Win_bytes_forward  256.0           1.091640
69         min_seg_size_forward   20.0           0.808011
7         Fwd Packet Length Max    6.0           0.668346


In [5]:
import json

feature_columns = X_train.columns.tolist()

with open("../models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

print(f"U ruajten {len(feature_columns)} feature columns")
print(feature_columns[:5], "...")

U ruajten 78 feature columns
['Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets'] ...


In [1]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")

Sesioni u rikthye!


In [1]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")

Sesioni u rikthye!


In [3]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")

Sesioni u rikthye!


In [6]:
import importlib
import app.services.llm_explainer as le
importlib.reload(le)

import inspect
print(inspect.getsource(le.generate_explanation))

def generate_explanation(predicted_label, confidence, top_shap_features):
    prompt = _build_prompt(predicted_label, confidence, top_shap_features)

    if settings.llm_provider == "claude":
        explanation_text, model_name = _generate_with_claude(prompt)
    else:
        explanation_text, model_name = _generate_with_gemini(prompt)

    return {
        "explanation_text": explanation_text,
        "llm_model": model_name,
        "llm_prompt_version": PROMPT_VERSION,
    }



In [10]:
importlib.reload(le)

result = le.generate_explanation(
    predicted_label="BENIGN",
    confidence=0.9999988079071045,
    top_shap_features=[
        {'feature': 'Bwd Packet Length Min', 'value': 127.0, 'shap_contribution': 1.9068595170974731},
        {'feature': 'Bwd Packet Length Mean', 'value': 127.0, 'shap_contribution': 1.2742890119552612},
        {'feature': 'Protocol', 'value': 17.0, 'shap_contribution': 1.0160354375839233},
        {'feature': 'Destination Port', 'value': 53.0, 'shap_contribution': 0.7084704041481018},
        {'feature': 'Init_Win_bytes_forward', 'value': -1.0, 'shap_contribution': 0.43682509660720825}
    ]
)

print(result['explanation_text'])
print(f"\nModel: {result['llm_model']}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [11]:
models_list = client.models.list()
for m in models_list:
    print(m.name)

NameError: name 'client' is not defined

In [12]:
from app.services.llm_explainer import _get_gemini_client

client = _get_gemini_client()
for m in client.models.list():
    if 'generateContent' in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-3.7-flash-video-understanding-eap
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-roboti

In [13]:
importlib.reload(le)

result = le.generate_explanation(
    predicted_label="BENIGN",
    confidence=0.9999988079071045,
    top_shap_features=[
        {'feature': 'Bwd Packet Length Min', 'value': 127.0, 'shap_contribution': 1.9068595170974731},
        {'feature': 'Bwd Packet Length Mean', 'value': 127.0, 'shap_contribution': 1.2742890119552612},
        {'feature': 'Protocol', 'value': 17.0, 'shap_contribution': 1.0160354375839233},
        {'feature': 'Destination Port', 'value': 53.0, 'shap_contribution': 0.7084704041481018},
        {'feature': 'Init_Win_bytes_forward', 'value': -1.0, 'shap_contribution': 0.43682509660720825}
    ]
)

print(result['explanation_text'])
print(f"\nModel: {result['llm_model']}")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [14]:
result = le.generate_explanation(
    predicted_label="BENIGN",
    confidence=0.9999988079071045,
    top_shap_features=[
        {'feature': 'Bwd Packet Length Min', 'value': 127.0, 'shap_contribution': 1.9068595170974731},
        {'feature': 'Bwd Packet Length Mean', 'value': 127.0, 'shap_contribution': 1.2742890119552612},
        {'feature': 'Protocol', 'value': 17.0, 'shap_contribution': 1.0160354375839233},
        {'feature': 'Destination Port', 'value': 53.0, 'shap_contribution': 0.7084704041481018},
        {'feature': 'Init_Win_bytes_forward', 'value': -1.0, 'shap_contribution': 0.43682509660720825}
    ]
)

print(result['explanation_text'])
print(f"\nModel: {result['llm_model']}")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [15]:
result = le.generate_explanation(
    predicted_label="BENIGN",
    confidence=0.9999988079071045,
    top_shap_features=[
        {'feature': 'Bwd Packet Length Min', 'value': 127.0, 'shap_contribution': 1.9068595170974731},
        {'feature': 'Bwd Packet Length Mean', 'value': 127.0, 'shap_contribution': 1.2742890119552612},
        {'feature': 'Protocol', 'value': 17.0, 'shap_contribution': 1.0160354375839233},
        {'feature': 'Destination Port', 'value': 53.0, 'shap_contribution': 0.7084704041481018},
        {'feature': 'Init_Win_bytes_forward', 'value': -1.0, 'shap_contribution': 0.43682509660720825}
    ]
)

print(result['explanation_text'])
print(f"\nModel: {result['llm_model']}")

Ky trafik është klasifikuar si plotësisht i sigurt dhe normal (**Benign**), pasi përfaqëson një komunikim standard të shërbimit DNS. Aktiviteti zhvillohet përmes protokollit UDP (Protokolli 17) drejt portës së zakonshme 53, me një madhësi konstante dhe tipike të paketave të përgjigjes prej 127 bajtësh. Gjithashtu, mungesa e parametrave të dritares TCP përputhet plotësisht me natyrën e këtij protokolli pa lidhje direkte (stateless). Si përfundim, kjo është një kërkesë e rregullt e rrjetit për zgjidhjen e emrave të domeneve dhe nuk paraqet asnjë rrezik.

Model: gemini-flash-latest


In [1]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")


Sesioni u rikthye!


In [2]:
import requests

attack_row = X_test[y_test != 'BENIGN'].iloc[0]
feature_dict = attack_row.to_dict()

ingest_response = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.5",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.95,
        "attackType": "DDoS",
        "flowTimestamp": "2026-08-14T12:00:00Z"
    }
)
print("Flow response:", ingest_response.json())

Flow response: {'id': '081ec562-b9b0-41cc-91d1-e077ba99675c', 'datasetSource': 'LAB_LIVE', 'sourceIp': '10.0.0.5', 'destinationIp': '10.0.0.100', 'sourcePort': 443, 'destinationPort': 80, 'protocol': 'TCP', 'featureVector': {'Destination Port': 80.0, 'Protocol': 6.0, 'Flow Duration': 5481877.0, 'Total Fwd Packets': 5.0, 'Total Backward Packets': 0.0, 'Total Length of Fwd Packets': 30.0, 'Total Length of Bwd Packets': 0.0, 'Fwd Packet Length Max': 6.0, 'Fwd Packet Length Min': 6.0, 'Fwd Packet Length Mean': 6.0, 'Fwd Packet Length Std': 0.0, 'Bwd Packet Length Max': 0.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 0.0, 'Bwd Packet Length Std': 0.0, 'Flow Bytes/s': 5.472578097, 'Flow Packets/s': 0.912096349, 'Flow IAT Mean': 1370469.25, 'Flow IAT Std': 2740283.205, 'Flow IAT Max': 5480894.0, 'Flow IAT Min': 0.0, 'Fwd IAT Total': 5481877.0, 'Fwd IAT Mean': 1370469.25, 'Fwd IAT Std': 2740283.205, 'Fwd IAT Max': 5480894.0, 'Fwd IAT Min': 0.0, 'Bwd IAT Total': 0.0, 'Bwd IAT Mean'

In [3]:
alarms_response = requests.get("http://localhost:8080/api/alarms?status=NEW")
alarms = alarms_response.json()
print(alarms)


[{'id': 'dd9d2f57-0af4-4b6b-9fd4-9ac00f3e9dcb', 'networkFlowId': '081ec562-b9b0-41cc-91d1-e077ba99675c', 'severity': 'CRITICAL', 'status': 'NEW', 'createdAt': '2026-08-19T18:14:48.433959Z', 'acknowledgedAt': None, 'resolvedAt': None}]


In [4]:
alarm_id = alarms[0]['id']

explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [5]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

200
{'predicted_label': 'DDoS', 'confidence': 0.9999984502792358, 'explanation_text': 'Sistemi ka identifikuar me siguri maksimale një sulm të llojit **DDoS** në këtë fluks trafiku. \n\nAlarmi u shkaktua sepse komunikimi përbëhet nga një numër shumë i kufizuar paketash me përmasa jashtëzakonisht të vogla (vetëm 30 bajtë të dhëna në total), çka nuk përkon me një trafik normal përdoruesi. Për më tepër, parametrat teknikë të TCP-së — si madhësia minimale e kreut (header) dhe dritarja fillestare tejet e ulët (256 bajtë) — janë tregues tipikë të paketave të programuara artificialisht nga vegla sulmi për të mbingarkuar shërbimin. Rekomandohet të verifikoni menjëherë IP-në burimore dhe të konsideroni bllokimin e saj në firewall.', 'llm_model': 'gemini-flash-latest', 'llm_prompt_version': 'v1'}


In [6]:
explanation_check = requests.get(f"http://localhost:8080/api/alarms/{alarm_id}/explanation")
print(explanation_check.json())

{'id': '528648ef-e667-41f9-84d0-324a82637f16', 'alarmId': 'dd9d2f57-0af4-4b6b-9fd4-9ac00f3e9dcb', 'explanationText': 'Sistemi ka identifikuar me siguri maksimale një sulm të llojit **DDoS** në këtë fluks trafiku. \n\nAlarmi u shkaktua sepse komunikimi përbëhet nga një numër shumë i kufizuar paketash me përmasa jashtëzakonisht të vogla (vetëm 30 bajtë të dhëna në total), çka nuk përkon me një trafik normal përdoruesi. Për më tepër, parametrat teknikë të TCP-së — si madhësia minimale e kreut (header) dhe dritarja fillestare tejet e ulët (256 bajtë) — janë tregues tipikë të paketave të programuara artificialisht nga vegla sulmi për të mbingarkuar shërbimin. Rekomandohet të verifikoni menjëherë IP-në burimore dhe të konsideroni bllokimin e saj në firewall.', 'llmModel': 'gemini-flash-latest', 'llmPromptVersion': 'v1', 'generatedAt': '2026-08-19T18:17:22.972569Z'}


In [8]:
attack_row2 = X_test[y_test != 'BENIGN'].iloc[1]  # nje shembull tjeter, ndryshe nga i pari
feature_dict2 = attack_row2.to_dict()

ingest_response2 = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.7",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict2.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict2,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.92,
        "attackType": "PortScan",
        "flowTimestamp": "2026-08-19T20:45:00Z"
    }
)
print("Flow response:", ingest_response2.json())

Flow response: {'id': '50c80db6-33a1-4448-937a-932d3fa99c20', 'datasetSource': 'LAB_LIVE', 'sourceIp': '10.0.0.7', 'destinationIp': '10.0.0.100', 'sourcePort': 443, 'destinationPort': 80, 'protocol': 'TCP', 'featureVector': {'Destination Port': 80.0, 'Protocol': 6.0, 'Flow Duration': 117028556.0, 'Total Fwd Packets': 7.0, 'Total Backward Packets': 7.0, 'Total Length of Fwd Packets': 356.0, 'Total Length of Bwd Packets': 11595.0, 'Fwd Packet Length Max': 356.0, 'Fwd Packet Length Min': 0.0, 'Fwd Packet Length Mean': 50.85714286, 'Fwd Packet Length Std': 134.5553524, 'Bwd Packet Length Max': 4344.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 1656.428571, 'Bwd Packet Length Std': 1946.27472, 'Flow Bytes/s': 102.1203748, 'Flow Packets/s': 0.119628922, 'Flow IAT Mean': 9002196.615, 'Flow IAT Std': 28000000.0, 'Flow IAT Max': 101000000.0, 'Flow IAT Min': 1.0, 'Fwd IAT Total': 117000000.0, 'Fwd IAT Mean': 19500000.0, 'Fwd IAT Std': 40400000.0, 'Fwd IAT Max': 101000000.0, 'Fwd IAT

In [9]:
attack_row2 = X_test[y_test != 'BENIGN'].iloc[1]  # nje shembull tjeter, ndryshe nga i pari
feature_dict2 = attack_row2.to_dict()

ingest_response2 = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.7",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict2.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict2,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.92,
        "attackType": "PortScan",
        "flowTimestamp": "2026-08-19T20:45:00Z"
    }
)
print("Flow response:", ingest_response2.json())

Flow response: {'id': '6615c718-603e-4ea8-b774-c7a083533a35', 'datasetSource': 'LAB_LIVE', 'sourceIp': '10.0.0.7', 'destinationIp': '10.0.0.100', 'sourcePort': 443, 'destinationPort': 80, 'protocol': 'TCP', 'featureVector': {'Destination Port': 80.0, 'Protocol': 6.0, 'Flow Duration': 117028556.0, 'Total Fwd Packets': 7.0, 'Total Backward Packets': 7.0, 'Total Length of Fwd Packets': 356.0, 'Total Length of Bwd Packets': 11595.0, 'Fwd Packet Length Max': 356.0, 'Fwd Packet Length Min': 0.0, 'Fwd Packet Length Mean': 50.85714286, 'Fwd Packet Length Std': 134.5553524, 'Bwd Packet Length Max': 4344.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 1656.428571, 'Bwd Packet Length Std': 1946.27472, 'Flow Bytes/s': 102.1203748, 'Flow Packets/s': 0.119628922, 'Flow IAT Mean': 9002196.615, 'Flow IAT Std': 28000000.0, 'Flow IAT Max': 101000000.0, 'Flow IAT Min': 1.0, 'Fwd IAT Total': 117000000.0, 'Fwd IAT Mean': 19500000.0, 'Fwd IAT Std': 40400000.0, 'Fwd IAT Max': 101000000.0, 'Fwd IAT

In [10]:
attack_row2 = X_test[y_test != 'BENIGN'].iloc[1]  # nje shembull tjeter, ndryshe nga i pari
feature_dict2 = attack_row2.to_dict()

ingest_response2 = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.7",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict2.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict2,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.92,
        "attackType": "PortScan",
        "flowTimestamp": "2026-08-19T20:45:00Z"
    }
)
print("Flow response:", ingest_response2.json())

Flow response: {'id': 'ee6ae8ff-b27d-40c9-a964-a150ff8db684', 'datasetSource': 'LAB_LIVE', 'sourceIp': '10.0.0.7', 'destinationIp': '10.0.0.100', 'sourcePort': 443, 'destinationPort': 80, 'protocol': 'TCP', 'featureVector': {'Destination Port': 80.0, 'Protocol': 6.0, 'Flow Duration': 117028556.0, 'Total Fwd Packets': 7.0, 'Total Backward Packets': 7.0, 'Total Length of Fwd Packets': 356.0, 'Total Length of Bwd Packets': 11595.0, 'Fwd Packet Length Max': 356.0, 'Fwd Packet Length Min': 0.0, 'Fwd Packet Length Mean': 50.85714286, 'Fwd Packet Length Std': 134.5553524, 'Bwd Packet Length Max': 4344.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 1656.428571, 'Bwd Packet Length Std': 1946.27472, 'Flow Bytes/s': 102.1203748, 'Flow Packets/s': 0.119628922, 'Flow IAT Mean': 9002196.615, 'Flow IAT Std': 28000000.0, 'Flow IAT Max': 101000000.0, 'Flow IAT Min': 1.0, 'Fwd IAT Total': 117000000.0, 'Fwd IAT Mean': 19500000.0, 'Fwd IAT Std': 40400000.0, 'Fwd IAT Max': 101000000.0, 'Fwd IAT

In [11]:
attack_row2 = X_test[y_test != 'BENIGN'].iloc[1]  # nje shembull tjeter, ndryshe nga i pari
feature_dict2 = attack_row2.to_dict()

ingest_response2 = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.7",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict2.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict2,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.92,
        "attackType": "PortScan",
        "flowTimestamp": "2026-08-19T20:45:00Z"
    }
)
print("Flow response:", ingest_response2.json())

Flow response: {'id': '4e0bb7bc-29f0-4a79-8b8a-2f3a0d019a3c', 'datasetSource': 'LAB_LIVE', 'sourceIp': '10.0.0.7', 'destinationIp': '10.0.0.100', 'sourcePort': 443, 'destinationPort': 80, 'protocol': 'TCP', 'featureVector': {'Destination Port': 80.0, 'Protocol': 6.0, 'Flow Duration': 117028556.0, 'Total Fwd Packets': 7.0, 'Total Backward Packets': 7.0, 'Total Length of Fwd Packets': 356.0, 'Total Length of Bwd Packets': 11595.0, 'Fwd Packet Length Max': 356.0, 'Fwd Packet Length Min': 0.0, 'Fwd Packet Length Mean': 50.85714286, 'Fwd Packet Length Std': 134.5553524, 'Bwd Packet Length Max': 4344.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 1656.428571, 'Bwd Packet Length Std': 1946.27472, 'Flow Bytes/s': 102.1203748, 'Flow Packets/s': 0.119628922, 'Flow IAT Mean': 9002196.615, 'Flow IAT Std': 28000000.0, 'Flow IAT Max': 101000000.0, 'Flow IAT Min': 1.0, 'Fwd IAT Total': 117000000.0, 'Fwd IAT Mean': 19500000.0, 'Fwd IAT Std': 40400000.0, 'Fwd IAT Max': 101000000.0, 'Fwd IAT

In [1]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")

ModuleNotFoundError: No module named 'dill'

In [2]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")

Sesioni u rikthye!


In [3]:
importances = xgb_smote.feature_importances_
feature_ranking = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("Top 15 features (sipas XGBoost built-in importance):")
print(feature_ranking.head(15))


Top 15 features (sipas XGBoost built-in importance):
                        feature  importance
0         Bwd Packet Length Min    0.329269
1                PSH Flag Count    0.106951
2              act_data_pkt_fwd    0.094589
3                      Idle Min    0.074004
4   Total Length of Fwd Packets    0.049000
5   Total Length of Bwd Packets    0.047343
6           Average Packet Size    0.044742
7                 Bwd Packets/s    0.043514
8         Bwd Packet Length Std    0.024538
9             Min Packet Length    0.021173
10                   Active Std    0.020591
11            Bwd Header Length    0.017471
12        Fwd Packet Length Min    0.015289
13        Fwd Packet Length Max    0.014795
14      Init_Win_bytes_backward    0.012454


In [7]:
def run_feature_subset_experiment(n_features, feature_ranking, X_train_full, y_train_full,
                                    X_test_full, y_test_full):
    selected_features = feature_ranking['feature'].head(n_features).tolist()

    X_train_subset = X_train_full[selected_features]
    X_test_subset = X_test_full[selected_features]

    model = xgb.XGBClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        eval_metric='mlogloss'
    )

    start = time.time()
    model.fit(X_train_subset, y_train_full)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test_subset)
    predict_time = time.time() - start

    accuracy = accuracy_score(y_test_full, y_pred)
    f1_macro = f1_score(y_test_full, y_pred, average='macro')
    precision_macro = precision_score(y_test_full, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test_full, y_pred, average='macro', zero_division=0)
    latency_ms = (predict_time / len(X_test_subset)) * 1000

    return {
        'n_features': n_features,
        'selected_features': selected_features,
        'model': model,
        'train_time': train_time,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'latency_ms': latency_ms,
    }

In [8]:
feature_counts_to_test = [10, 20, 30, 50, 78]
rq3_results = []

for n in feature_counts_to_test:
    print(f"Duke testuar me {n} features...")
    result = run_feature_subset_experiment(
        n_features=n,
        feature_ranking=feature_ranking,
        X_train_full=X_train_balanced,
        y_train_full=y_train_balanced_encoded,
        X_test_full=X_test,
        y_test_full=y_test_encoded
    )
    rq3_results.append(result)
    print(f"  Accuracy: {result['accuracy']:.4f} | F1 macro: {result['f1_macro']:.4f} | "
          f"Latency: {result['latency_ms']:.4f} ms | Train time: {result['train_time']:.1f}s\n")

print("Te gjitha eksperimentet perfunduan.")

Duke testuar me 10 features...
  Accuracy: 0.9594 | F1 macro: 0.6873 | Latency: 0.0050 ms | Train time: 32.5s

Duke testuar me 20 features...
  Accuracy: 0.9893 | F1 macro: 0.7851 | Latency: 0.0046 ms | Train time: 36.6s

Duke testuar me 30 features...
  Accuracy: 0.9968 | F1 macro: 0.8280 | Latency: 0.0049 ms | Train time: 46.2s

Duke testuar me 50 features...
  Accuracy: 0.9988 | F1 macro: 0.8794 | Latency: 0.0053 ms | Train time: 67.5s

Duke testuar me 78 features...
  Accuracy: 0.9988 | F1 macro: 0.8876 | Latency: 0.0060 ms | Train time: 95.6s

Te gjitha eksperimentet perfunduan.


In [9]:
summary_df = pd.DataFrame([{
    'n_features': r['n_features'],
    'accuracy': r['accuracy'],
    'f1_macro': r['f1_macro'],
    'precision_macro': r['precision_macro'],
    'recall_macro': r['recall_macro'],
    'latency_ms': r['latency_ms'],
    'train_time_s': r['train_time'],
} for r in rq3_results])

print(summary_df.to_string(index=False))

 n_features  accuracy  f1_macro  precision_macro  recall_macro  latency_ms  train_time_s
         10  0.959364  0.687340         0.671401      0.813893    0.004972     32.455559
         20  0.989269  0.785098         0.747796      0.897106    0.004571     36.635901
         30  0.996803  0.827990         0.797280      0.906234    0.004941     46.216742
         50  0.998801  0.879436         0.868014      0.899928    0.005345     67.458675
         78  0.998815  0.887587         0.876183      0.907048    0.005978     95.587256


In [1]:
import sys
sys.path.append("..")

import dill
dill.load_session('../notebooks/session_checkpoint.pkl')
print("Sesioni u rikthye!")

Sesioni u rikthye!


In [2]:
try:
    print(f"rq3_results ekziston, ka {len(rq3_results)} eksperimente")
except NameError:
    print("rq3_results S'EKZISTON - duhet ta rikrijojme")

rq3_results S'EKZISTON - duhet ta rikrijojme


In [3]:
importances = xgb_smote.feature_importances_
feature_ranking = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("Top 15 features:")
print(feature_ranking.head(15))

Top 15 features:
                        feature  importance
0         Bwd Packet Length Min    0.329269
1                PSH Flag Count    0.106951
2              act_data_pkt_fwd    0.094589
3                      Idle Min    0.074004
4   Total Length of Fwd Packets    0.049000
5   Total Length of Bwd Packets    0.047343
6           Average Packet Size    0.044742
7                 Bwd Packets/s    0.043514
8         Bwd Packet Length Std    0.024538
9             Min Packet Length    0.021173
10                   Active Std    0.020591
11            Bwd Header Length    0.017471
12        Fwd Packet Length Min    0.015289
13        Fwd Packet Length Max    0.014795
14      Init_Win_bytes_backward    0.012454


In [4]:
def run_feature_subset_experiment(n_features, feature_ranking, X_train_full, y_train_full,
                                    X_test_full, y_test_full):
    selected_features = feature_ranking['feature'].head(n_features).tolist()

    X_train_subset = X_train_full[selected_features]
    X_test_subset = X_test_full[selected_features]

    model = xgb.XGBClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        eval_metric='mlogloss'
    )

    start = time.time()
    model.fit(X_train_subset, y_train_full)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test_subset)
    predict_time = time.time() - start

    accuracy = accuracy_score(y_test_full, y_pred)
    f1_macro = f1_score(y_test_full, y_pred, average='macro')
    precision_macro = precision_score(y_test_full, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test_full, y_pred, average='macro', zero_division=0)
    latency_ms = (predict_time / len(X_test_subset)) * 1000

    return {
        'n_features': n_features,
        'selected_features': selected_features,
        'model': model,
        'train_time': train_time,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'latency_ms': latency_ms,
    }

In [5]:
feature_counts_to_test = [10, 20, 30, 50, 78]
rq3_results = []

for n in feature_counts_to_test:
    print(f"Duke testuar me {n} features...")
    result = run_feature_subset_experiment(
        n_features=n,
        feature_ranking=feature_ranking,
        X_train_full=X_train_balanced,
        y_train_full=y_train_balanced_encoded,
        X_test_full=X_test,
        y_test_full=y_test_encoded
    )
    rq3_results.append(result)
    print(f"  Accuracy: {result['accuracy']:.4f} | F1 macro: {result['f1_macro']:.4f} | "
          f"Latency: {result['latency_ms']:.4f} ms | Train time: {result['train_time']:.1f}s\n")

print("Te gjitha eksperimentet perfunduan.")

Duke testuar me 10 features...
  Accuracy: 0.9594 | F1 macro: 0.6873 | Latency: 0.0051 ms | Train time: 31.4s

Duke testuar me 20 features...
  Accuracy: 0.9893 | F1 macro: 0.7851 | Latency: 0.0045 ms | Train time: 36.4s

Duke testuar me 30 features...
  Accuracy: 0.9968 | F1 macro: 0.8280 | Latency: 0.0050 ms | Train time: 46.2s

Duke testuar me 50 features...
  Accuracy: 0.9988 | F1 macro: 0.8794 | Latency: 0.0057 ms | Train time: 67.8s

Duke testuar me 78 features...
  Accuracy: 0.9988 | F1 macro: 0.8876 | Latency: 0.0121 ms | Train time: 103.9s

Te gjitha eksperimentet perfunduan.


In [6]:
rq3_model_ids = {}

for r in rq3_results:
    n = r['n_features']

    model_path = models_dir / f"xgb_smote_top{n}features_v1.joblib"
    joblib.dump(r['model'], model_path)

    model_response = register_model(
        algorithm="XGBoost",
        name=f"xgb-smote-top{n}features-v1",
        trained_on_dataset="CICIDS2017",
        artifact_path=f"models/xgb_smote_top{n}features_v1.joblib",
        trained_at_iso=datetime.now(timezone.utc).isoformat(),
        hyperparameters='{"n_estimators": 100, "random_state": 42, "tree_method": "hist"}',
        feature_set=f"top_{n}_features_by_importance"
    )
    model_id = model_response["id"]
    rq3_model_ids[n] = model_id

    experiment_response = record_experiment_result(
        ml_model_id=model_id,
        tested_on_dataset="CICIDS2017",
        accuracy=r['accuracy'],
        precision=r['precision_macro'],
        recall=r['recall_macro'],
        f1=r['f1_macro'],
        avg_latency_ms=r['latency_ms'],
        sample_size=len(X_test),
        feature_set_used=f"top_{n}_features",
        notes=f"RQ3 - feature selection experiment, top {n} features (nga 78 totale)"
    )
    print(f"n={n}: model={model_id}, experiment={experiment_response['id']}")

print("\nTe gjitha eksperimentet RQ3 u regjistruan ne Spring Boot.")

n=10: model=9a776ceb-a0c7-4cfe-85fb-f47a28c1d31e, experiment=6efe5f24-cdcf-4775-b21f-12f6a86c65e6
n=20: model=0ab5e744-3369-4b6e-a40b-4b13938a9829, experiment=bd16940c-fe93-43d1-a3a5-c13cbb085cb2
n=30: model=ea59f364-d518-4231-8c32-a5c46ca15033, experiment=7632f4b3-7042-4a2e-ae31-177a0e5c6507
n=50: model=bc1e1b60-480e-4630-9864-42d66d462fa5, experiment=ce71b895-b25f-4d59-8d2d-ebdf10628d87
n=78: model=22c197af-1d3e-4f78-9bc8-cf7546b9a4a7, experiment=3e292f1d-68f3-4933-810c-c8e55d661361

Te gjitha eksperimentet RQ3 u regjistruan ne Spring Boot.


In [7]:
dill.dump_session('../notebooks/session_checkpoint.pkl')
print("Checkpoint u perditesua me eksperimentet RQ3!")

Checkpoint u perditesua me eksperimentet RQ3!


In [8]:
attack_row = X_test[y_test != 'BENIGN'].iloc[2]  # tjeter shembull, i pandryshuar deri tani
feature_dict = attack_row.to_dict()

ingest_response = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.9",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.97,
        "attackType": "PortScan",
        "flowTimestamp": "2026-08-21T11:00:00Z"
    }
)
print("Flow response:", ingest_response.json())

NameError: name 'requests' is not defined

In [9]:
import requests


In [10]:
attack_row = X_test[y_test != 'BENIGN'].iloc[2]  # tjeter shembull, i pandryshuar deri tani
feature_dict = attack_row.to_dict()

ingest_response = requests.post(
    "http://localhost:8080/api/alarms/ingest",
    json={
        "sourceIp": "10.0.0.9",
        "destinationIp": "10.0.0.100",
        "sourcePort": 443,
        "destinationPort": int(feature_dict.get("Destination Port", 80)),
        "protocol": "TCP",
        "featureVector": feature_dict,
        "predictedLabel": "ATTACK",
        "predictionConfidence": 0.97,
        "attackType": "PortScan",
        "flowTimestamp": "2026-08-21T11:00:00Z"
    }
)
print("Flow response:", ingest_response.json())

Flow response: {'id': '76b7613f-618c-45b1-a272-b93409cbdbbf', 'datasetSource': 'LAB_LIVE', 'sourceIp': '10.0.0.9', 'destinationIp': '10.0.0.100', 'sourcePort': 443, 'destinationPort': 80, 'protocol': 'TCP', 'featureVector': {'Destination Port': 80.0, 'Protocol': 6.0, 'Flow Duration': 98812759.0, 'Total Fwd Packets': 6.0, 'Total Backward Packets': 6.0, 'Total Length of Fwd Packets': 417.0, 'Total Length of Bwd Packets': 11595.0, 'Fwd Packet Length Max': 417.0, 'Fwd Packet Length Min': 0.0, 'Fwd Packet Length Mean': 69.5, 'Fwd Packet Length Std': 170.2395371, 'Bwd Packet Length Max': 4344.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 1932.5, 'Bwd Packet Length Std': 1977.812908, 'Flow Bytes/s': 121.5632487, 'Flow Packets/s': 0.121441807, 'Flow IAT Mean': 8982978.091, 'Flow IAT Std': 29800000.0, 'Flow IAT Max': 98800000.0, 'Flow IAT Min': 1.0, 'Fwd IAT Total': 98800000.0, 'Fwd IAT Mean': 19800000.0, 'Fwd IAT Std': 44200000.0, 'Fwd IAT Max': 98800000.0, 'Fwd IAT Min': 1.0, 'Bw

In [11]:
alarm_id = ingest_response.json()  # kjo eshte flow, jo alarm - duhet ta marrim alarm ID

In [12]:
alarms_response = requests.get("http://localhost:8080/api/alarms?status=NEW")
alarms = alarms_response.json()
latest_alarm = alarms[0]  # alarmi me i ri
print(latest_alarm)

{'id': 'e57aee56-064d-448b-9865-b9cd1b9c6e35', 'networkFlowId': '76b7613f-618c-45b1-a272-b93409cbdbbf', 'severity': 'CRITICAL', 'status': 'NEW', 'createdAt': '2026-08-21T09:20:07.764767Z', 'acknowledgedAt': None, 'resolvedAt': None}


In [13]:
alarm_id = latest_alarm['id']

explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [14]:
alarm_id = latest_alarm['id']

explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [15]:
alarm_id = latest_alarm['id']

explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

200
{'predicted_label': 'DoS Hulk', 'confidence': 0.9999995231628418, 'explanation_text': 'Sistemi i sigurisë ka identifikuar me siguri të plotë një sulm të llojit **DoS Hulk**, i cili synon të mbingarkojë dhe të nxjerrë jashtë funksioni serverin tuaj të uebit. Ky alarm u nxit sepse trafiku i dyshimtë shfaq pauza të gjata pasiviteti dhe luhatje të mëdha në kohën e mbërritjes së paketave, një sjellje tipike e këtij sulmi për të shmangur filtrat standardë. Për më tepër, përgjigjet e serverit tregojnë një madhësi shumë të vogël të dritares TCP (TCP Window), gjë që sinjalizon se shërbimi po përjeton mbingarkesë dhe po ka vështirësi në përpunimin e kërkesave. Rekomandohet të kontrolloni menjëherë performancën e serverit të synuar dhe të bllokoni IP-në burimore të këtij trafiku.', 'llm_model': 'gemini-flash-latest', 'llm_prompt_version': 'v1'}


In [16]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [17]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [18]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [19]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [20]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [21]:
explain_response = requests.post(
    "http://localhost:8000/api/explain",
    json={
        "alarm_id": alarm_id,
        "feature_vector": feature_dict
    }
)
print(explain_response.status_code)
print(explain_response.json())

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)